# MMF GPU Models on Serverless GPU
Auto-generated by mmf-agent. Runs all models of one GPU class (global DL, foundation or foundationforecast) in this notebook on serverless GPU compute (one A10, environment AI v6). `serverless=True` makes foundation and foundationforecast models predict on the driver's GPU, because Spark workers on serverless have no GPU.

In [ ]:
%pip install {pip_install_args} --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
train_table = "{train_table}"
freq = "{freq}"
prediction_length = {prediction_length}
backtest_length = {backtest_length}
stride = {stride}
metric = "{metric}"
active_models = {active_models}
group_id = "{group_id}"
date_col = "{date_col}"
target = "{target}"

# experiment_path is passed by the job task (Skill 4 Step 5b base_parameters). Skill 4 Step 1c
# pre-creates it and checks that it doesn't overlap the notebook folder.
dbutils.widgets.text("experiment_path", "{experiment_path}")
experiment_path = dbutils.widgets.get("experiment_path").strip()
assert experiment_path, (
    "experiment_path widget is empty. Skill 4 Step 5b must set it as a base_parameter on the job task."
)

# Covariates — see Skill 4 Feature Type Decision Guide.
# SAFE:  static_features, dynamic_historical_*. dynamic_future_* needs a scoring table with known
# future values for every future ds; if missing, the pipeline errors or drops series. FF* models
# that support covariates use only dynamic_future_*.
static_features = {static_features}
dynamic_future_numerical = {dynamic_future_numerical}
dynamic_future_categorical = {dynamic_future_categorical}
dynamic_historical_numerical = {dynamic_historical_numerical}
dynamic_historical_categorical = {dynamic_historical_categorical}
# Scoring table: "" (same as train_table) unless dynamic_future_* is in use.
scoring_table = "{scoring_table}"

In [ ]:
import uuid
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_forecast


def _exog_kwargs():
    kw = {}
    if static_features:
        kw["static_features"] = static_features
    if dynamic_future_numerical:
        kw["dynamic_future_numerical"] = dynamic_future_numerical
    if dynamic_future_categorical:
        kw["dynamic_future_categorical"] = dynamic_future_categorical
    if dynamic_historical_numerical:
        kw["dynamic_historical_numerical"] = dynamic_historical_numerical
    if dynamic_historical_categorical:
        kw["dynamic_historical_categorical"] = dynamic_historical_categorical
    return kw


train_fqn = catalog + "." + schema + "." + train_table
_st = (scoring_table or "").strip()
scoring_fqn = train_fqn if not _st else catalog + "." + schema + "." + _st

# One run_id for all models of this class, as in the classic orchestrator.
run_id = str(uuid.uuid4())
total = len(active_models)
print("Running " + str(total) + " models, run_id " + run_id)

# One run_forecast call per model, so each model's checkpoint is released before the next loads.
for i, model in enumerate(active_models, 1):
    print("\n[" + str(i) + "/" + str(total) + "] Running model: " + model)
    run_forecast(
        spark=spark,
        train_data=train_fqn,
        scoring_data=scoring_fqn,
        scoring_output=catalog + "." + schema + "." + use_case + "_scoring_output",
        evaluation_output=catalog + "." + schema + "." + use_case + "_evaluation_output",
        model_output=catalog + "." + schema,
        group_id=group_id,
        date_col=date_col,
        target=target,
        freq=freq,
        prediction_length=prediction_length,
        backtest_length=backtest_length,
        stride=stride,
        metric=metric,
        train_predict_ratio=1,
        data_quality_check=False,
        resample=False,
        active_models=[model],
        experiment_path=experiment_path,
        use_case_name=use_case,
        run_id=run_id,
        accelerator="gpu",
        serverless=True,
        **_exog_kwargs(),
    )
    print("[" + str(i) + "/" + str(total) + "] Completed model: " + model)

print("\nAll " + str(total) + " models completed.")